[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdouSalamSisawo/TensorOrbit-EDA-Bootcamp/blob/main/day-1-foundations-pandas/day_1_notebook.ipynb)

# TensorOrbit EDA Bootcamp — Day 1
### Foundations of Data Science, Python & Pandas

**Instructor:** Abdou Salam Sisawo
**Flow:** Python Fundamentals → Meet Pandas → Working With the Messy Employee Dataset → Guided EDA Challenge

This notebook follows the exact same journey as the Day 1 slide deck, in the same order, so you can
code along live or work through it afterwards on your own. Every section starts with **why** a tool
matters before showing **how** to use it — don't just run the cells, read the questions each one answers.

> 💡 **Tip:** Run cells top to bottom with `Shift + Enter`. If something errors, it's almost always
> because an earlier cell wasn't run yet — scroll up and run from the top.

## 📂 About the dataset

This notebook loads the **Messy Employee Dataset** directly from the TensorOrbit GitHub repo, so
there is nothing to download or upload — it works out of the box in **Google Colab**.

- **Repo copy (used by this notebook):** https://github.com/AbdouSalamSisawo/TensorOrbit-EDA-Bootcamp/blob/main/day-1-foundations-pandas/materials/Messy_Employee_dataset.csv
- **Original Kaggle listing:** https://www.kaggle.com/datasets/desolution01/messy-employee-dataset

If you've cloned the repo and are running this notebook **locally** from inside it, the loader
cell in Part 3 also falls back to the local copy of the CSV automatically — no internet required
once you've cloned it.

---
# Part 1 — Python Fundamentals for Data Analysis

We are **not** learning all of Python here — just enough to read and write Pandas code confidently.
Every idea in this section reappears inside Pandas in Part 3.

## 1.1 Variables & Data Types

*How do we store a value in Python?*

In [ ]:
name = "Alice"
age = 25
salary = 50000
remote = True

print(name, age, salary, remote)
print(type(name), type(age), type(salary), type(remote))

- `str` = text, `int` = whole number, `float` = decimal, `bool` = True/False, `None` = nothing
- Python figures out the type automatically — no need to declare it up front

## 1.2 Think Before You Code: `"25"` vs `25`

`"25"` and `25` look the same to us. Are they the same to Python?

- Quotes = text (`str`) · No quotes = number (`int`)
- This exact mix-up is why a `Salary` column full of text can't be averaged in Pandas later

In [ ]:
a = "25"
b = 25

print(a == b)          # are they equal?
print(type(a), type(b))

try:
    print(a + 5)
except TypeError as e:
    print("Error:", e)   # this is exactly what happens when a numeric column is stored as text

## 1.3 Operators

In [ ]:
age = 35
salary = 62000
remote = True

# Arithmetic:  +  -  *  /  //  %  **
print(age + 5, salary * 12)

# Comparison:  ==  !=  >  <  >=  <=
print(age > 30)
print(salary >= 50000)

# Logical:  and  or  not
print(age > 30 and remote == True)

Remember that last line — we'll write this **exact same logic** to filter an entire spreadsheet
of employees a little later.

## 1.4 Strings

In [ ]:
name = "Abdou"

print(name[0])            # indexing — first character
print(name[0:3])          # slicing — first three characters

print(name.lower())
print(name.strip())
print(f"Employee: {name}")   # f-string — combine text and variables

Messy real-world names, emails, and job titles need exactly these tools — later we'll use the
`.str` versions of these same methods on an entire Pandas column at once.

## 1.5 Lists

In [ ]:
employees = ["Alice", "Bob", "Charlie"]

print(employees[0])
print(employees[0:2])

employees.append("David")
print(employees)

for employee in employees:
    print(employee)

A list is like one column of values sitting in order — that's exactly what a table column is.

## 1.6 Dictionaries

In [ ]:
employee = {
    "name": "Alice",
    "age": 25,
    "department": "HR"
}

print(employee["name"])
print(employee)

A DataFrame row behaves a lot like a dictionary — the column name is the key.

## 1.7 Conditions

In [ ]:
salary = 62000

if salary > 50000:
    print("High salary")
else:
    print("Other")

Same 'true or false' logic — later we apply it to a whole column at once as a **Pandas filter**.

## 1.8 Loops

In [ ]:
employees = ["Alice", "Bob", "Charlie"]

for employee in employees:
    print(employee)

for i in range(5):
    print(i)

Pandas will do this kind of repetition for us automatically — but it helps to see it manually once.

## 1.9 Functions

In [ ]:
def calculate_total_salary(salary, bonus):
    return salary + bonus

print(calculate_total_salary(50000, 2000))

## 1.10 Imports — The Bridge to Pandas

This is the exact line every notebook you write from now on will start with.

In [ ]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

---
# Part 2 — Meet Pandas

Pandas is a Python library built for working with structured, tabular data — rows and columns,
just like Excel. Nearly every EDA task from here on starts with Pandas.

## 2.1 Series — One Column of Data

In [ ]:
s = pd.Series([10, 20, 30])
s

A Series is a single labelled column of values — think of it as a list, but with an index attached.

## 2.2 DataFrame — A Full Table

In [ ]:
sample_df = pd.DataFrame({
    "name": ["Alice", "Bob"],
    "age": [25, 30]
})
sample_df

- Rows = records (one employee per row)
- Columns = variables (name, age, department...)
- Notice the shape: a **dictionary of lists** becomes a table — the exact same key → value
  pattern from section 1.6, now holding entire columns instead of single values.

---
# Part 3 — Working With the Messy Employee Dataset

**Dataset:** Messy Employee Dataset (synthetic HR data — **1,020 rows, 12 columns**)
**Source:** https://www.kaggle.com/datasets/desolution01/messy-employee-dataset
**Repo copy:** `day-1-foundations-pandas/materials/Messy_Employee_dataset.csv`

Columns: `Employee_ID`, `First_Name`, `Last_Name`, `Age`, `Department_Region`, `Status`,
`Join_Date`, `Salary`, `Email`, `Phone`, `Performance_Score`, `Remote_Work`

Real data-quality issues you'll actually find in this file as we go:
- Missing `Age` (~21% of rows) and a smaller number of missing `Salary` values
- A compound `Department_Region` column (e.g. `"DevOps-California"`) mixing two variables together
- No single "full name" column — just `First_Name` and `Last_Name` separately
- `Phone` values that are large negative numbers rather than anything resembling a real phone number
- Categorical fields worth knowing about up front: `Status` has **three** values
  (`Active` / `Pending` / `Inactive`), and `Performance_Score` is **text**
  (`Poor` / `Average` / `Good` / `Excellent`), not a 1–5 number

> The goal isn't "can you run Pandas?" — it's **"can you look at data and figure out what needs
> to happen next?"** Not every dataset is messy in the exact same way, which is exactly why we
> always inspect first instead of assuming.

## 3.0 Load the Data

In [ ]:
import pandas as pd
import os

# Primary source: fetch straight from the GitHub repo (works anywhere with internet, incl. Colab)
RAW_URL = (
    "https://raw.githubusercontent.com/AbdouSalamSisawo/TensorOrbit-EDA-Bootcamp/"
    "main/day-1-foundations-pandas/materials/Messy_Employee_dataset.csv"
)

# Fallback: local paths, in case you're offline but have the repo cloned
LOCAL_CANDIDATES = [
    "Messy_Employee_dataset.csv",
    "materials/Messy_Employee_dataset.csv",
    "day-1-foundations-pandas/materials/Messy_Employee_dataset.csv",
]

df = None
try:
    df = pd.read_csv(RAW_URL)
    print(f"Loaded {df.shape[0]} rows directly from GitHub:\n{RAW_URL}")
except Exception as e:
    print(f"Could not fetch from GitHub ({e}).\nFalling back to a local copy...")
    for fname in LOCAL_CANDIDATES:
        if os.path.exists(fname):
            df = pd.read_csv(fname)
            print(f"Loaded local file '{fname}' instead.")
            break

if df is None:
    raise FileNotFoundError(
        "Could not load the dataset from GitHub or find it locally.\n"
        "Check your internet connection, or place the CSV at one of: "
        f"{LOCAL_CANDIDATES}"
    )

`pd.read_excel("file.xlsx")` works the same way for Excel files. `df` is now our entire
spreadsheet, living in memory as a DataFrame.

## 3.1 First Look at the Data — Shape & Structure

| Question | Tool |
|---|---|
| How big is my dataset? | `df.shape` |
| What variables do I have? | `df.columns` |
| What do the first rows look like? | `df.head()` |
| What about the last rows? | `df.tail()` |

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumns:")
print(list(df.columns))

In [ ]:
df.head()

In [ ]:
df.tail()

## 3.2 First Look at the Data — Types & Summary

| Question | Tool |
|---|---|
| What are the data types & missing values? | `df.info()` |
| What do the numeric columns look like? | `df.describe()` |

In [ ]:
df.info()

In [ ]:
df.describe()

👀 **Look closely at `Phone` above.** The min/mean/max are all large *negative* numbers — that's
not what a real phone number looks like. Not every data-quality problem is a missing value or a
wrong dtype; sometimes a column is technically numeric but **doesn't mean anything real**. There's
no reliable way to recover genuine phone numbers from this, so the honest move is to flag it (as
we just did) rather than pretend we can fix it — and treat the column as an opaque identifier if
we ever need it, not as something to do arithmetic on.

## 3.3 Selecting Data

In [ ]:
df["Salary"].head()

In [ ]:
df[["First_Name", "Last_Name", "Salary", "Department_Region"]].head()

In [ ]:
# .loc  -> select by label (row label, column name)
print(df.loc[0, "Salary"])

# .iloc -> select by position (row number, column number) — column 3 is Age
print(df.columns[3], "->", df.iloc[0, 3])

## 3.4 Filtering Data

Remember: `age > 30 and remote == True` from Part 1 — same logic, now applied to an entire
column instead of one variable. Note the syntax difference for multiple conditions:
use `&` instead of `and`, and wrap each condition in its own parentheses.

In [ ]:
df[df["Age"] > 40].head()

In [ ]:
df[df["Status"] == "Active"].head()

In [ ]:
df[(df["Age"] > 30) & (df["Remote_Work"] == True)].head()

## 3.5 Sorting Data

*Who are the highest-paid employees?*

In [ ]:
df.sort_values("Salary").head()

In [ ]:
df.sort_values("Salary", ascending=False).head()

In [ ]:
df.nlargest(10, "Salary")

## 3.6 Descriptive Statistics

- What is the average employee age?
- What is the median salary?
- How many employees are Active vs Pending vs Inactive?
- What is the highest salary in the dataset?

In [ ]:
print("Average age:", round(df["Age"].mean(), 1))
print("Median salary:", df["Salary"].median())

In [ ]:
df["Status"].value_counts()

## 3.7 Missing Values

Missing data is **not automatically "bad"** — first we investigate:
- How much is missing? Which columns? Why might it be missing? What should we do?

`fillna()` replaces gaps; `dropna()` removes rows entirely — the right choice depends on the question.

In [ ]:
df.isnull().sum()

In [ ]:
# Example: fill missing Age with the median age (a common, defensible default)
age_filled = df["Age"].fillna(df["Age"].median())
print("Missing before:", df["Age"].isnull().sum())
print("Missing after: ", age_filled.isnull().sum())

In [ ]:
# Example: drop rows that are missing ANY value (use with care — inspect first!)
df_complete_rows_only = df.dropna()
print("Rows before dropna:", len(df))
print("Rows after dropna: ", len(df_complete_rows_only))

## 3.8 Duplicates

If the same employee appears twice, counts and averages become misleading.

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].head(10)

In [ ]:
df_no_duplicates = df.drop_duplicates()
print("Rows before:", len(df))
print("Rows after: ", len(df_no_duplicates))

## 3.9 Data Types & Conversion

`df.dtypes` tells us what Pandas thinks each column holds. In this dataset, `Salary` and `Age`
already loaded as real numbers, and `Remote_Work` loaded as a real boolean — but `Join_Date` is
still plain text, so date-specific tools (like `.dt.year`) won't work on it yet.

In [ ]:
df.dtypes

In [ ]:
df["Join_Date_Parsed"] = pd.to_datetime(df["Join_Date"], errors="coerce")

print("Unparseable Join_Date values:", df["Join_Date_Parsed"].isnull().sum())
df[["Join_Date", "Join_Date_Parsed"]].head()

**Not every dataset is this clean, though.** It's very common for a `Salary`-style column to
arrive as text — things like `"$85,000"` or `"N/A"` — which breaks `.mean()` exactly like the
`"25" + 5` error from section 1.2. The defensive pattern below (`pd.to_numeric(..., errors="coerce")`)
is worth knowing even though *this particular* dataset doesn't need it for `Salary`:

In [ ]:
# A small illustrative example — not our real data, just the general technique
messy_salary_example = pd.Series(["85000", "$92,000", "N/A", "110000"])
cleaned_example = pd.to_numeric(
    messy_salary_example.str.replace(r"[^0-9.\-]", "", regex=True),
    errors="coerce",
)
print(cleaned_example.tolist())

**One more type worth adjusting:** `Phone` loaded as a number (`int64`), but we already saw it
doesn't represent anything real. Casting it to text makes that intent explicit — it's an
identifier-like field, not a quantity:

In [ ]:
df["Phone"] = df["Phone"].astype(str)
df["Phone"].head()

## 3.10 Creating & Modifying Columns

New columns can be built from existing ones — this is called **feature creation**. There's no
single "full name" column in this dataset, only `First_Name` and `Last_Name` — a perfect
opportunity to build one ourselves.

In [ ]:
df["Full_Name"] = df["First_Name"] + " " + df["Last_Name"]
df[["First_Name", "Last_Name", "Full_Name"]].head()

In [ ]:
# Salary here reads as an annual figure (values sit in the ~$50k–$120k range),
# so a natural derived column is the monthly equivalent
df["Monthly_Salary"] = df["Salary"] / 12
df[["Salary", "Monthly_Salary"]].head()

**Bonus — splitting a compound column:** `Department_Region` mixes two variables together
(e.g. `"DevOps-California"`). We can split it into two clean columns using the same `.str` tools
from the next section:

In [ ]:
split_cols = df["Department_Region"].astype(str).str.split("-", n=1, expand=True)
df["Department"] = split_cols[0]
df["Region"] = split_cols[1]
df[["Department_Region", "Department", "Region"]].head()

## 3.11 String Operations

The `.str` accessor applies string methods to an entire column at once — the exact same
`.strip()` / `.lower()` methods from section 1.4, now running on every row simultaneously.

In [ ]:
df["Full_Name"] = df["Full_Name"].astype(str).str.strip()
df["Full_Name_Lower"] = df["Full_Name"].str.lower()
df[["Full_Name", "Full_Name_Lower"]].head()

In [ ]:
looks_like_gmail = df["Email"].astype(str).str.contains("@gmail", case=False, na=False)
print("Employees with a gmail address:", looks_like_gmail.sum())

Every email in this dataset actually uses the same placeholder domain (`@example.com`), so that
count above is expected to be `0` — the point isn't the result, it's that you can't know that
without checking. On a real HR export you'd often find a genuine mix of work and personal domains.

## 3.12 Working With Dates

*What year did most employees join?*

In [ ]:
df["Join_Year"] = df["Join_Date_Parsed"].dt.year
df["Join_Month"] = df["Join_Date_Parsed"].dt.month
df["Join_Weekday"] = df["Join_Date_Parsed"].dt.day_name()

df[["Join_Date_Parsed", "Join_Year", "Join_Month", "Join_Weekday"]].head()

In [ ]:
df["Join_Year"].value_counts().sort_index()

## 3.13 GroupBy & Aggregation

`groupby` = **split → apply → combine**. This is one of the most powerful tools in Pandas.

- Which department/region has the highest average salary?
- How many employees are in each status?
- What is the average salary by performance score?

In [ ]:
df.groupby("Department")["Salary"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("Status")["Salary"].agg(["count", "mean", "min", "max"])

In [ ]:
df.groupby("Performance_Score")["Salary"].mean().sort_values(ascending=False)

## 3.14 Merge & Concat

`concat` stacks datasets on top of each other; `merge` joins them side-by-side using a
shared key column. Here we build a small department lookup table and merge it back onto
our employee data, purely to demonstrate the pattern.

In [ ]:
departments = pd.DataFrame({
    "Department": df["Department"].dropna().unique(),
})
departments["Department_Head_Count_Target"] = range(10, 10 + len(departments))
departments

In [ ]:
merged = pd.merge(df, departments, on="Department", how="left")
merged[["Full_Name", "Department", "Department_Head_Count_Target"]].head()

In [ ]:
# concat example: stacking two halves of the dataset back together
half_1 = df.iloc[: len(df) // 2]
half_2 = df.iloc[len(df) // 2 :]
stacked = pd.concat([half_1, half_2])
print("Original rows:", len(df), "| Stacked rows:", len(stacked))

## 3.15 Exporting Cleaned Data

Analysis usually ends by producing a clean dataset for someone else — a colleague, a
dashboard, or a model.

In [ ]:
df.to_csv("cleaned_employee_data.csv", index=False)
print("Saved cleaned_employee_data.csv with", df.shape[0], "rows and", df.shape[1], "columns.")

---
# Part 4 — Guided EDA Challenge

Your turn. Work through each question below **using the tools from Part 3**. Think before you
type — for each one, ask yourself *"what would I do first?"* before writing any code.

Use `df` (the added columns like `Salary`, `Join_Date_Parsed`, `Department`, `Region`
are available) or go back to the original raw columns — your choice.

**1.** How many employees are in the dataset?

In [ ]:
# Your code here

**2.** How many columns are there?

In [ ]:
# Your code here

**3.** Which columns contain missing values?

In [ ]:
# Your code here

**4.** How many duplicate records exist?

In [ ]:
# Your code here

**5.** What is the average age?

In [ ]:
# Your code here

**6.** What is the median salary?

In [ ]:
# Your code here

**7.** Which employees have the highest salaries?

In [ ]:
# Your code here

**8.** What is the Active/Inactive status distribution?

In [ ]:
# Your code here

**9.** Which department-region has the highest average salary?

In [ ]:
# Your code here

**10.** What is the split between remote and non-remote employees?

In [ ]:
# Your code here

**11.** What problems do you notice with Join_Date?

In [ ]:
# Your code here

**12.** What problems do you notice with Salary?

In [ ]:
# Your code here

**13.** What data-cleaning steps would you recommend overall?

In [ ]:
# Your code here

---
# Part 5 — Recap & What's Next

**Today we covered:**
- ✓ Python fundamentals for data analysis (variables, types, operators, strings, lists,
  dictionaries, conditions, loops, functions, imports)
- ✓ Pandas basics — Series & DataFrame
- ✓ Loading and inspecting a real messy dataset
- ✓ Selecting, filtering, and sorting data
- ✓ Descriptive statistics
- ✓ Handling missing values and duplicates
- ✓ Fixing data types (text-stored numbers, inconsistent dates)
- ✓ Creating and modifying columns, string cleaning, datetime parts
- ✓ GroupBy & aggregation
- ✓ Merge/concat and exporting cleaned data

> **"Don't just clean the data. Ask questions about it."**

**Coming up:**
- **Day 2:** NumPy + Matplotlib
- **Day 3:** Applied EDA + Capstone project

— Abdou Salam Sisawo · TensorOrbit EDA Bootcamp